# Fase 3: Preparación y Modelado de Datos (ETL)
Este notebook contiene el proceso ETL (Extracción, Transformación y Carga) para preparar los datos de Churn para su uso en PowerBI o Tableau.

## 1. Importación de Librerías

In [1]:
import pandas as pd
import numpy as np
import os

## 2. EXTRACCIÓN (Extract)
Cargamos el archivo original `Churn_Modelling.csv`.

In [2]:
input_file = "Churn_Modelling.csv"
output_file = "Churn_Modelling_Cleaned.csv"

print(f"Cargando datos desde {input_file}...")
try:
    df = pd.read_csv(input_file)
    print(f"Dataset original cargado exitosamente: {df.shape[0]} filas y {df.shape[1]} columnas.")
except FileNotFoundError:
    print(f"Error: No se encontró el archivo {input_file}.")
    
df.head()

Cargando datos desde Churn_Modelling.csv...
Dataset original cargado exitosamente: 10000 filas y 14 columnas.


,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


## 3. TRANSFORMACIÓN (Transform)
### 3.1 Limpieza Básica
Eliminamos columnas que no aportan valor analítico por su alta cardinalidad o por ser identificadores únicos. También tratamos nulos y duplicados.

In [3]:
columnas_a_eliminar = ['RowNumber', 'CustomerId', 'Surname']
df.drop(columns=[col for col in columnas_a_eliminar if col in df.columns], inplace=True)

# Tratamiento de valores nulos (Imputación con mediana y moda)
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(include=['object']).columns

df[num_cols] = df[num_cols].fillna(df[num_cols].median())
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])
    
# Eliminar registros duplicados
df.drop_duplicates(inplace=True)

C:\Users\DiegoRdrz\AppData\Local\Temp\ipykernel_13004\520173935.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


### 3.2 Tratamiento de Outliers y Conversión de Tipos
Aplicamos una técnica de capping suave utilizando el Rango Intercuartílico (IQR) para atenuar el efecto de valores extremos en CreditScore, y lo convertimos a entero.

In [4]:
if 'CreditScore' in df.columns:
    Q1 = df['CreditScore'].quantile(0.25)
    Q3 = df['CreditScore'].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df['CreditScore'] = np.where(df['CreditScore'] < lower_bound, lower_bound, df['CreditScore'])
    df['CreditScore'] = np.where(df['CreditScore'] > upper_bound, upper_bound, df['CreditScore'])
    
    # Conversión a entero
    df['CreditScore'] = df['CreditScore'].astype(int)

### 3.3 Formateo de Variables para BI y Preservación de Variables Numéricas Originales
PowerBI y Tableau funcionan mejor con variables categóricas descriptivas en inglés. Además, se mantienen las variables binarias originales (con sufijo `_num`) para posteriores pruebas de hipótesis.

In [5]:
if 'HasCrCard' in df.columns:
    df['HasCrCard_num'] = df['HasCrCard']
    df['HasCrCard'] = df['HasCrCard'].map({1: 'Yes', 0: 'No'})
    
if 'IsActiveMember' in df.columns:
    df['IsActiveMember_num'] = df['IsActiveMember']
    df['IsActiveMember'] = df['IsActiveMember'].map({1: 'Yes', 0: 'No'})
    
if 'Exited' in df.columns:
    df['Exited_num'] = df['Exited']
    df['Exited'] = df['Exited'].map({1: 'Churn', 0: 'Retained'})

### 3.4 Creación de Dimensiones (Categorización)
Agrupamos variables continuas en categorías (solo etiquetas, sin rangos, en inglés) para facilitar la creación de gráficos y filtros en los dashboards.

In [6]:
# Grupos de Edad
if 'Age' in df.columns:
    bins_age = [0, 30, 50, 150]
    labels_age = ['Young', 'Adult', 'Senior']
    df['AgeGroup'] = pd.cut(df['Age'], bins=bins_age, labels=labels_age, right=False)

# Grupos de Credit Score
if 'CreditScore' in df.columns:
    bins_score = [0, 500, 650, 750, 900]
    labels_score = ['Poor', 'Fair', 'Good', 'Excellent']
    df['CreditScoreGroup'] = pd.cut(df['CreditScore'], bins=bins_score, labels=labels_score, right=False)

# Grupos de Saldo (Balance)
if 'Balance' in df.columns:
    bins_balance = [-1, 0, 100000, float('inf')]
    labels_balance = ['No Balance', 'Medium', 'High']
    df['BalanceGroup'] = pd.cut(df['Balance'], bins=bins_balance, labels=labels_balance, right=True)

df[['Age', 'AgeGroup', 'CreditScore', 'CreditScoreGroup', 'Balance', 'BalanceGroup']].head()

,Age,AgeGroup,CreditScore,CreditScoreGroup,Balance,BalanceGroup
0,42,Adult,619,Fair,0.00,No Balance
1,41,Adult,608,Fair,83807.86,Medium
2,42,Adult,502,Fair,159660.80,High
3,39,Adult,699,Good,0.00,No Balance
4,43,Adult,850,Excellent,125510.82,High


## 4. CARGA (Load)
Exportamos el dataset final limpio a CSV.

In [7]:
print(f"Exportando datos limpios a {output_file}...")
try:
    df.to_csv(output_file, index=False, encoding='utf-8')
    print(f"¡Dataset final guardado exitosamente en {os.path.abspath(output_file)}!")
except Exception as e:
    print(f"Error al guardar el archivo: {e}")

df.head()

Exportando datos limpios a Churn_Modelling_Cleaned.csv...
¡Dataset final guardado exitosamente en C:\Users\DiegoRdrz\Desktop\code\suicide\World_Suicide_rates\Entrega 2\Churn_Modelling_Cleaned.csv!


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,HasCrCard_num,IsActiveMember_num,Exited_num,AgeGroup,CreditScoreGroup,BalanceGroup
0,619,France,Female,42,2,0.00,1,Yes,Yes,101348.88,Churn,1,1,1,Adult,Fair,No Balance
1,608,Spain,Female,41,1,83807.86,1,No,Yes,112542.58,Retained,0,1,0,Adult,Fair,Medium
2,502,France,Female,42,8,159660.80,3,Yes,No,113931.57,Churn,1,0,1,Adult,Fair,High
3,699,France,Female,39,1,0.00,2,No,No,93826.63,Retained,0,0,0,Adult,Good,No Balance
4,850,Spain,Female,43,2,125510.82,1,Yes,Yes,79084.10,Retained,1,1,0,Adult,Excellent,High
